# Complete Image Classification Experiment

This notebook contains the project code in **one `.ipynb` file**.

It includes the custom baseline model, Inception-style architecture, weight initialization utilities, training helpers, weight saving/loading, and the ImageNet-pretrained InceptionV3 experiment.


In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt

In [17]:
def transfer_model(
    input_shape,
    backbone='efficientnet',
    trainable=False,
    last_layer_unit=1,
    last_layer_activation='sigmoid'
):

    inputs = tf.keras.layers.Input(shape=input_shape)

    if backbone == 'resnet50':
        base = tf.keras.applications.ResNet50(
            include_top=False,
            weights='imagenet',
            input_shape=input_shape
        )

    elif backbone == 'efficientnet':
        base = tf.keras.applications.EfficientNetB0(
            include_top=False,
            weights='imagenet',
            input_shape=input_shape
        )

    elif backbone == 'mobilenet':
        base = tf.keras.applications.MobileNetV2(
            include_top=False,
            weights='imagenet',
            input_shape=input_shape
        )

    else:
        raise ValueError("Unknown backbone")

    base.trainable = trainable

    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    outputs = tf.keras.layers.Dense(
        last_layer_unit,
        activation=last_layer_activation
    )(x)

    model = tf.keras.Model(inputs, outputs)

    return model

In [ ]:
# InceptionV3 is created inside inception_v3_model() when needed.
# This keeps model construction separate from model training.


In [3]:
def data_augmentation(
    rotation=0.2,
    zoom=0.2,
    brightness=0.2,
    contrast=0.2,
    horizontal_flip=True
):
    layers = []

    if horizontal_flip:
        layers.append(tf.keras.layers.RandomFlip("horizontal"))

    layers.extend([
        tf.keras.layers.RandomRotation(rotation),
        tf.keras.layers.RandomZoom(zoom),
        tf.keras.layers.RandomBrightness(brightness),
        tf.keras.layers.RandomContrast(contrast),
    ])

    return tf.keras.Sequential(layers)

data_agg = data_augmentation()

In [4]:
def load_data(
    directory,
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    seed=42
):
    train_data = tf.keras.utils.image_dataset_from_directory(
        directory,
        image_size=image_size,
        batch_size=batch_size,
        validation_split=validation_split,
        subset="training",
        seed=seed,
        label_mode="binary"
    )

    val_data = tf.keras.utils.image_dataset_from_directory(
        directory,
        image_size=image_size,
        batch_size=batch_size,
        validation_split=validation_split,
        subset="validation",
        seed=seed,
        label_mode="binary"
    )

    return train_data, val_data

In [5]:
def prepare_datasets(train_data, val_data, buffer_size=1000):
    train_batches = (train_data.shuffle(buffer_size).prefetch(tf.data.AUTOTUNE))
    val_batches = val_data.prefetch(tf.data.AUTOTUNE)

    return train_batches, val_batches

In [6]:
def image_show(image, title=None):
    if hasattr(image, 'numpy'):
        image = image.numpy()

    image = image.astype('uint8')

    plt.imshow(image)
    plt.axis('off')

    if title is not None:
        plt.title(title)

    plt.show()

In [ ]:
def inception_resnet_block(x, filters, conv_regularizer=None):
    b1 = tf.keras.layers.Conv2D(
        filters, (1, 1),
        padding='same',
        activation='relu',
        kernel_regularizer=conv_regularizer
    )(x)

    b2 = tf.keras.layers.Conv2D(
        filters, (1, 1),
        padding='same',
        activation='relu',
        kernel_regularizer=conv_regularizer
    )(x)
    b2 = tf.keras.layers.Conv2D(
        filters, (3, 3),
        padding='same',
        activation='relu',
        kernel_regularizer=conv_regularizer
    )(b2)

    b3 = tf.keras.layers.Conv2D(
        filters, (1, 1),
        padding='same',
        activation='relu',
        kernel_regularizer=conv_regularizer
    )(x)
    b3 = tf.keras.layers.Conv2D(
        filters, (5, 5),
        padding='same',
        activation='relu',
        kernel_regularizer=conv_regularizer
    )(b3)

    merged = tf.keras.layers.Concatenate(axis=-1)([b1, b2, b3])

    merged = tf.keras.layers.Conv2D(
        x.shape[-1],
        (1, 1),
        padding='same',
        kernel_regularizer=conv_regularizer
    )(merged)

    outputs = tf.keras.layers.Add()([x, merged])
    outputs = tf.keras.layers.ReLU()(outputs)

    return outputs


In [ ]:
def base_model(
    input_shape,
    conv_regularizer=None,
    dense_regularizer=None,
    last_layer_unit=1,
    last_layer_activation='sigmoid'
):
    inputs = tf.keras.layers.Input(shape=input_shape)

    x = data_agg(inputs)
    x = tf.keras.layers.Rescaling(1.0 / 255)(x)

    x = tf.keras.layers.Conv2D(
        64, (3, 3),
        padding='same',
        activation='relu',
        kernel_regularizer=conv_regularizer
    )(x)

    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    x = inception_resnet_block(x, 128, conv_regularizer)
    x = inception_resnet_block(x, 128, conv_regularizer)

    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    x = inception_resnet_block(x, 128, conv_regularizer)
    x = inception_resnet_block(x, 128, conv_regularizer)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dense(
        1024,
        activation='relu',
        kernel_regularizer=dense_regularizer
    )(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    x = tf.keras.layers.Dense(
        512,
        activation='relu',
        kernel_regularizer=dense_regularizer
    )(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    outputs = tf.keras.layers.Dense(
        last_layer_unit,
        activation=last_layer_activation
    )(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name='base_model')


In [ ]:
def inception_v3_model(
    input_shape,
    trainable=False,
    dense_regularizer=None,
    last_layer_unit=1,
    last_layer_activation='sigmoid'
):
    inputs = tf.keras.layers.Input(shape=input_shape)

    x = data_agg(inputs)
    x = tf.keras.applications.inception_v3.preprocess_input(x)

    base = tf.keras.applications.InceptionV3(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )

    base.trainable = trainable

    x = base(inputs=x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dense(
        1024,
        activation='relu',
        kernel_regularizer=dense_regularizer
    )(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    x = tf.keras.layers.Dense(
        512,
        activation='relu',
        kernel_regularizer=dense_regularizer
    )(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    x = tf.keras.layers.Dense(
        128,
        activation='relu',
        kernel_regularizer=dense_regularizer
    )(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    outputs = tf.keras.layers.Dense(
        last_layer_unit,
        activation=last_layer_activation
    )(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name='inception_v3_model')


In [29]:
class EarlyStopping(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs=None):
    if logs.get('accuracy',0)>=0.95:
      self.model.stop_training = True
      print('We have acchived the desigred accuracy!')

In [ ]:
# Example model construction.
# Keep the experiments separate so you can compare them fairly.

model_base = base_model(input_shape=(224, 224, 3))

model_inception_v3 = inception_v3_model(
    input_shape=(224, 224, 3),
    trainable=False
)

def compiling(
    model,
    optimizer=None,
    loss=None
):
    if optimizer is None:
        optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    if loss is None:
        loss = tf.keras.losses.BinaryCrossentropy()

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['accuracy']
    )


In [ ]:
model_inception_v3.summary()


In [ ]:
model_base.summary()


In [ ]:
def training(
    model,
    train_data,
    val_data,
    epochs=50,
    early_stopping=False,
    weights_path='weights/model.weights.h5'
):
    callbacks = []

    if early_stopping:
        callbacks.append(EarlyStopping())

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        callbacks=callbacks
    )

    Path(weights_path).parent.mkdir(parents=True, exist_ok=True)
    model.save_weights(weights_path)

    return history


def load_model_weights(
    model,
    weights_path
):
    model.load_weights(weights_path)
    return model


## Experiment plan

1. Train `base_model` from scratch as the baseline.
2. Save its weights separately.
3. Train `inception_v3_model` with ImageNet initialization as a separate transfer-learning experiment.
4. If testing weight initialization from the custom base model into another custom architecture, transfer only layers whose type and weight shapes are compatible.
5. Compare the models using the same data split, augmentation, optimizer settings, and evaluation metrics.


In [ ]:
def copy_compatible_layer_weights(source_model, target_model, layer_pairs):
    """Copy weights only between explicitly matched, shape-compatible layers."""
    for source_name, target_name in layer_pairs:
        source_layer = source_model.get_layer(source_name)
        target_layer = target_model.get_layer(target_name)

        source_weights = source_layer.get_weights()
        target_weights = target_layer.get_weights()

        if len(source_weights) != len(target_weights):
            raise ValueError(
                f'Weight-count mismatch: {source_name} -> {target_name}'
            )

        if any(sw.shape != tw.shape for sw, tw in zip(source_weights, target_weights)):
            raise ValueError(
                f'Weight-shape mismatch: {source_name} -> {target_name}'
            )

        target_layer.set_weights(source_weights)

    return target_model


### Important

The custom `base_model` weights cannot be blindly loaded into Keras `InceptionV3`. The architectures and layer shapes are different. The helper above is intentionally explicit: only compatible layer pairs can be initialized from the base model.


In [ ]:
# Loading weights:
#
# 1. Recreate the SAME architecture/configuration.
# 2. Load the saved weights.
#
# Example:
# model_base = base_model(input_shape=(224, 224, 3))
# model_base = load_model_weights(model_base, 'weights/base_model.weights.h5')
#
# For a complete saved model (architecture + weights), use model.save('model.keras')
# and tf.keras.models.load_model('model.keras') instead.
